# Networking RAG - LangGraph Pipeline & Evaluation

This notebook builds the Networking RAG pipeline using LangGraph, utilizing flexible LLM configurations and evaluates it using RAGAS.

In [11]:
try:
    !pip install -q -r requirements.txt
    print("Dependencies installed successfully!")
except Exception as e:
    print(f"Installation Error: {e}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [12]:
try:
    import os
    import pickle
    import json
    import chromadb
    import pandas as pd
    import dotenv
    from typing import TypedDict, Optional, Any, Dict, List
    from datetime import datetime

    # LangGraph imports
    from langgraph.graph import StateGraph, START, END

    # Google AI imports
    from google import genai
    from google.colab import userdata, drive
    from google.genai.errors import APIError

    # Groq imports
    from groq import Groq

    # RAGAS imports
    from datasets import Dataset
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

    # LangChain imports for evaluation
    from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

    print("All libraries imported successfully!")

except Exception as e:
    print(f"Import Error: {e}")

All libraries imported successfully!


/tmp/ipykernel_4875/3735278441.py:25: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/tmp/ipykernel_4875/3735278441.py:25: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/tmp/ipykernel_4875/3735278441.py:25: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfu

In [13]:
try:
    drive.mount('/content/drive')
    print("Google Drive mounted successfully!")
except Exception as e:
    print(f"Drive Mount Error: {e}")

Mounted at /content/drive
Google Drive mounted successfully!


In [14]:
try:
    # Load Gemini API Key
    GEMINI_API_KEY = userdata.get('Gemini_key')
    if not GEMINI_API_KEY:
        raise ValueError("GEMINI_API_KEY not found in Colab secrets")
    os.environ['GOOGLE_API_KEY'] = GEMINI_API_KEY

    # Load Groq API Key
    GROQ_API_KEY = userdata.get('Groq_Key')
    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY not found in Colab secrets")
    os.environ['GROQ_API_KEY'] = GROQ_API_KEY

    print("API Keys loaded successfully!")

except Exception as e:
    print(f"API Key Error: {e}")
    print("Please ensure you have set Gemini_key and Groq_Key in Colab secrets")

API Keys loaded successfully!


Initialize API Clients

In [15]:
try:
    # Initialize Gemini Client
    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    print("Gemini Client initialized successfully!")

except Exception as e:
    print(f"Warning: Failed to initialize Gemini Client: {e}")
    gemini_client = None

try:
    # Initialize Groq Client
    groq_client = Groq(api_key=GROQ_API_KEY)
    print("Groq Client initialized successfully!")

except Exception as e:
    print(f"Warning: Failed to initialize Groq Client: {e}")
    groq_client = None

Gemini Client initialized successfully!
Groq Client initialized successfully!


 Define Configuration Constants

In [16]:
try:
    import os

    # Path to your ChromaDB on Google Drive
    CHROMA_DB_PATH = "/content/drive/MyDrive/RAG_Internship/chroma_db"

    # Path to evaluation data
    EVAL_DATA_PATH = "/content/drive/MyDrive/RAG_Internship/arrays.txt"

    # Path to wiki documents (optional, for reference)
    WIKI_DOCS_PATH = "/content/drive/MyDrive/RAG_Internship/wiki_docs"

    # RAG Prompt Template
    RAG_PROMPT_TEMPLATE = """You are a networking and AI assistant.

Answer only using the provided context.
If the answer is not available in the context, respond with:
"I could not find that information in the knowledge base."

Context:
{context}

Question:
{question}
"""

    # Verify ChromaDB exists
    if os.path.exists(CHROMA_DB_PATH):
        print(f"ChromaDB found at: {CHROMA_DB_PATH}")

        # Check if it has data (optional)
        import chromadb
        try:
            temp_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
            collections = temp_client.list_collections()
            if collections:
                print(f"   Collections found: {[c.name for c in collections]}")
                # Try to get count from wiki_rag collection
                try:
                    wiki_collection = temp_client.get_collection("wiki_rag")
                    print(f"   Documents in wiki_rag: {wiki_collection.count()}")
                except:
                    print("   'wiki_rag' collection not found yet")
            else:
                print("   No collections found. You may need to run ingestion first.")
        except:
            print("   Could not read ChromaDB details.")
    else:
        print(f"ChromaDB not found at: {CHROMA_DB_PATH}")
        print(" You need to run your Ingestion notebook first to create it.")

    print("\nConfiguration loaded successfully!")

except Exception as e:
    print(f"Configuration Error: {e}")

ChromaDB found at: /content/drive/MyDrive/RAG_Internship/chroma_db
   Collections found: ['wiki_rag']
   Documents in wiki_rag: 733

Configuration loaded successfully!


In [17]:
try:
    PROJECT_DIR = "/content/drive/MyDrive/RAG_Internship"
    print(f" PROJECT_DIR set to: {PROJECT_DIR}")
except Exception as e:
    print(f" Error: {e}")

 PROJECT_DIR set to: /content/drive/MyDrive/RAG_Internship


State Definition

In [18]:
try:
    class GraphState(TypedDict):
        question: str
        query_embedding: Optional[List[float]]
        retrieved_chunks: Optional[List[str]]
        retrieved_docs: Optional[List[Any]]
        context: Optional[str]
        answer: Optional[str]
        error: Optional[str]
        # Flexible config parameters
        llm_client: Optional[Any]
        llm_model: Optional[str]
        prompt_template: Optional[str]

    print("GraphState defined successfully!")

except Exception as e:
    print(f"State Definition Error: {e}")

GraphState defined successfully!


Initialize ChromaDB from Drive

In [19]:
try:
    chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

    # Try to get existing collection
    try:
        collection = chroma_client.get_collection(name="wiki_rag")
        print(f"Found existing collection: wiki_rag")
    except:
        # Create new collection if it doesn't exist
        collection = chroma_client.create_collection(name="wiki_rag")
        print("Created new collection: wiki_rag")

    print(f"   Collection count: {collection.count()} documents")

except Exception as e:
    print(f"ChromaDB Initialization Error: {e}")
    collection = None

Found existing collection: wiki_rag
   Collection count: 733 documents


Node 1 - Embed Question

In [20]:
def embed_question_node(state: GraphState) -> GraphState:
    try:
        question = state.get("question")
        if not question:
            raise ValueError("Question not found in state.")

        if gemini_client is None:
            raise RuntimeError("Gemini client is uninitialized.")

        response = gemini_client.models.embed_content(
            model="models/gemini-embedding-001",
            contents=question
        )
        state["query_embedding"] = response.embeddings[0].values

    except Exception as e:
        print(f"Error in embed_question_node: {e}")
        state["query_embedding"] = []
        state["error"] = str(e)

    return state

Node 2 - Retrieve Chunks

In [21]:
def retrieve_chunks_node(state: GraphState) -> GraphState:
    try:
        embedding = state.get("query_embedding")
        if not embedding:
            raise ValueError("No query embedding found in state.")

        if collection is None:
            raise RuntimeError("ChromaDB collection is uninitialized.")

        results = collection.query(
            query_embeddings=[embedding],
            n_results=5
        )

        state["retrieved_chunks"] = results.get("documents", [[]])[0]
        state["retrieved_docs"] = results.get("metadatas", [[]])[0]

    except Exception as e:
        print(f"Error in retrieve_chunks_node: {e}")
        state["retrieved_chunks"] = []
        state["error"] = str(e)

    return state

Node 3 - Generate Answer

In [22]:
def generate_answer_node(state: GraphState) -> GraphState:
    try:
        chunks = state.get("retrieved_chunks", [])
        context = "\n\n".join(chunks) if chunks else "No context available."
        question = state.get("question", "")

        # Pick flexible prompt template, client and model
        prompt_tpl = state.get("prompt_template") or RAG_PROMPT_TEMPLATE
        client_obj = state.get("llm_client") or groq_client
        model = state.get("llm_model") or "llama-3.3-70b-versatile"

        prompt = prompt_tpl.format(context=context, question=question)
        state["context"] = context

        # Support Groq / OpenAI compatible client structure
        if hasattr(client_obj, "chat") and hasattr(client_obj.chat, "completions"):
            response = client_obj.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}]
            )
            state["answer"] = response.choices[0].message.content

        # Support Google GenAI Client structure
        elif hasattr(client_obj, "models") and hasattr(client_obj.models, "generate_content"):
            response = client_obj.models.generate_content(
                model=model,
                contents=prompt
            )
            state["answer"] = response.text
        else:
            raise RuntimeError("Unsupported or uninitialized LLM client.")

    except Exception as e:
        print(f"Error in generate_answer_node: {e}")
        state["answer"] = "Error generating response from LLM."
        state["error"] = str(e)

    return state

LangGraph Workflow

In [23]:
try:
    graph_builder = StateGraph(GraphState)
    graph_builder.add_node("embed_question", embed_question_node)
    graph_builder.add_node("retrieve_chunks", retrieve_chunks_node)
    graph_builder.add_node("generate_answer", generate_answer_node)

    # Add edges
    graph_builder.add_edge(START, "embed_question")
    graph_builder.add_edge("embed_question", "retrieve_chunks")
    graph_builder.add_edge("retrieve_chunks", "generate_answer")
    graph_builder.add_edge("generate_answer", END)

    rag_graph = graph_builder.compile()
    print("LangGraph RAG pipeline compiled successfully!")

except Exception as e:
    print(f"Error compiling LangGraph: {e}")
    rag_graph = None

LangGraph RAG pipeline compiled successfully!


RAG Pipeline Function

In [24]:
def run_rag_pipeline(question: str, client_obj: Any = None, model: str = None) -> Dict:
    try:
        if rag_graph is None:
            raise RuntimeError("LangGraph RAG Pipeline is not compiled.")

        input_state = {
            "question": question,
            "llm_client": client_obj,
            "llm_model": model
        }
        return rag_graph.invoke(input_state)

    except Exception as e:
        print(f"Error running RAG pipeline: {e}")
        return {
            "question": question,
            "answer": "Error.",
            "retrieved_chunks": [],
            "error": str(e)
        }

Test the above function

In [25]:
try:
    if rag_graph:
        # Test with a simple question
        result = run_rag_pipeline("What is machine learning?")

        print(f"\n{'-'*60}")
        print(" Test Query: What is machine learning?")
        print(f"{'-'*60}")
        print(f"\n Answer:\n{result.get('answer', 'No answer generated')}")
        print(f"\n Retrieved Chunks: {len(result.get('retrieved_chunks', []))}")

        if result.get('error'):
            print(f"\n Error: {result.get('error')}")

except Exception as e:
    print(f"Test Error: {e}")


------------------------------------------------------------
 Test Query: What is machine learning?
------------------------------------------------------------

 Answer:
Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed.

 Retrieved Chunks: 5


 Load Evaluation Data

In [26]:
def load_evaluation_data(path: str = None) -> tuple:
    try:
        if path is None:
            path = EVAL_DATA_PATH

        if not os.path.exists(path):
            raise FileNotFoundError(f"{path} does not exist.")

        # Load lists by executing arrays.txt context safely
        globals_dict = {}
        with open(path, "r") as f:
            exec(f.read(), globals_dict)

        questions = globals_dict.get("test_questions", [])
        ground_truths = globals_dict.get("ground_truths", [])

        print(f" Loaded {len(questions)} questions and {len(ground_truths)} ground truths.")
        return questions, ground_truths

    except Exception as e:
        print(f" Error loading evaluation data: {e}")
        return [], []

# Load the data
try:
    test_questions, ground_truths = load_evaluation_data()
except Exception as e:
    print(f" Could not load evaluation data: {e}")
    test_questions = []
    ground_truths = []

 Loaded 20 questions and 20 ground truths.


RAGAS Evaluation Function

Using GROQ

In [28]:
# evaluate_rag_system with Groq
from datetime import datetime
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from langchain_groq import ChatGroq
from langchain_google_genai import GoogleGenerativeAIEmbeddings

def evaluate_rag_system(questions: List[str], answers: List[str], contexts: List[List[str]],
                        ground_truths: List[str], config_name: str) -> Optional[Dict]:
    """
    Evaluate RAG system using RAGAS metrics.
    Uses Groq for evaluation (more reliable than Gemini).
    """
    try:
        print(f" Starting RAGAS evaluation for {config_name}...")

        #  Use Groq for evaluation (more reliable, faster)
        evaluator_llm = ChatGroq(
            model="llama-3.3-70b-versatile",
            api_key=GROQ_API_KEY,
            temperature=0
        )
        print(f"    Evaluator LLM: Groq (llama-3.3-70b-versatile)")

        # Use Gemini for embeddings (embeddings work fine with Gemini)
        evaluator_embeddings = GoogleGenerativeAIEmbeddings(
            model="models/gemini-embedding-001",
            google_api_key=GEMINI_API_KEY
        )
        print(f"    Embeddings: Gemini (models/gemini-embedding-001)")

        # Prepare dataset
        data_dict = {
            "question": questions,
            "answer": answers,
            "contexts": contexts,
            "ground_truth": ground_truths
        }
        dataset = Dataset.from_dict(data_dict)
        print(f"    Dataset prepared: {len(dataset)} samples")

        # Run evaluation
        results = evaluate(
            dataset,
            metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
            llm=evaluator_llm,
            embeddings=evaluator_embeddings
        )
        print(f"    Evaluation completed!")

        # Convert to DataFrame and save
        df = results.to_pandas()
        safe_name = config_name.lower().replace(" ", "_")
        csv_filename = f"{PROJECT_DIR}/ragas_results_{safe_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        df.to_csv(csv_filename, index=False)
        print(f" Saved detailed results to {csv_filename}")

        return results

    except Exception as e:
        print(f" RAGAS Evaluation Error: {e}")
        print(f"   Error type: {type(e).__name__}")
        import traceback
        traceback.print_exc()
        return None

print(" evaluate_rag_system function updated to use Groq!")

 evaluate_rag_system function updated to use Groq!


/tmp/ipykernel_4875/270930735.py:5: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/tmp/ipykernel_4875/270930735.py:5: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/tmp/ipykernel_4875/270930735.py:5: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfulness,

In [29]:
def collect_rag_predictions(questions: List[str], client_obj: Any = None, model: str = None) -> tuple:
    """
    Collect predictions from the RAG pipeline for a list of questions.
    """
    answers = []
    contexts = []
    errors = []

    try:
        total = len(questions)
        for i, q in enumerate(questions):
            print(f" Processing {i+1}/{total}: {q[:50]}...")
            res = run_rag_pipeline(q, client_obj=client_obj, model=model)

            answers.append(res.get("answer", ""))
            contexts.append(res.get("retrieved_chunks", []))
            errors.append(res.get("error", None))

        print(f" Collected predictions for {len(questions)} questions.")
        return answers, contexts, errors

    except Exception as e:
        print(f" Error collecting predictions: {e}")
        return [], [], []

In [30]:
# run_full_evaluation_flow
def run_full_evaluation_flow(limit: int = None, configs: List[Dict] = None):
    """
    Evaluates multiple RAG configurations using RAGAS metrics.
    """
    try:
        if configs is None:
            configs = [
                {"name": "Groq-Llama-70B", "client": groq_client, "model": "llama-3.3-70b-versatile"},
                {"name": "Gemini-2.5-Flash", "client": gemini_client, "model": "gemini-2.5-flash"}
            ]

        # Use test data
        questions = test_questions[:limit] if limit else test_questions
        gts = ground_truths[:limit] if limit else ground_truths

        if not questions:
            print(" No questions found. Please check arrays.txt file.")
            return

        print(f"\n Evaluating {len(questions)} questions across {len(configs)} configurations...")
        print(f"   Evaluator: Groq (llama-3.3-70b-versatile)")
        print("-" * 60)

        comparison_results = {}

        for config in configs:
            cfg_name = config["name"]
            client_obj = config["client"]
            model_name = config["model"]

            print(f"\n{'='*60}")
            print(f" Evaluating Configuration: {cfg_name}")
            print(f"{'='*60}")

            # Collect predictions from the RAG pipeline
            answers, contexts, errors = collect_rag_predictions(
                questions, client_obj=client_obj, model=model_name
            )

            # Run evaluation using Groq
            results = evaluate_rag_system(
                questions, answers, contexts, gts, config_name=cfg_name
            )

            if results:
                comparison_results[cfg_name] = results

        # Print summary
        print(f"\n{'='*60}")
        print(" RAGAS CONFIGURATION SUMMARY")
        print(f"{'='*60}")
        for name, score in comparison_results.items():
            print(f"\n🔹 {name}:")
            if hasattr(score, 'to_dict'):
                try:
                    score_dict = score.to_dict()
                    for metric, value in score_dict.items():
                        if not metric.startswith('_'):
                            if isinstance(value, (int, float)):
                                print(f"   {metric}: {value:.4f}")
                            else:
                                print(f"   {metric}: {value}")
                except:
                    print(f"   {score}")
            else:
                print(f"   {score}")
        print(f"\n{'='*60}")

    except Exception as e:
        print(f" Evaluation Flow Error: {e}")
        import traceback
        traceback.print_exc()

print(" run_full_evaluation_flow function updated!")

 run_full_evaluation_flow function updated!


test with one question

In [ ]:
# Cell 17e: Test evaluation with 1 question
try:
    print(" Testing evaluation with 1 question...")
    print("-" * 60)

    # Test data
    test_q = ["What is machine learning?"]
    test_a = ["Machine learning is a field of study in artificial intelligence."]
    test_c = [["Machine learning is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data."]]
    test_gt = ["Machine learning is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data."]

    # Run test
    result = evaluate_rag_system(
        questions=test_q,
        answers=test_a,
        contexts=test_c,
        ground_truths=test_gt,
        config_name="Test"
    )

    if result:
        print("\n Test successful!")
        print(f"   Results: {result}")
    else:
        print("\n Test failed - returned None")

except Exception as e:
    print(f" Test Error: {e}")
    import traceback
    traceback.print_exc()

 Testing evaluation with 1 question...
------------------------------------------------------------
 Starting RAGAS evaluation for Test...
    Evaluator LLM: Groq (llama-3.3-70b-versatile)
    Embeddings: Gemini (models/gemini-embedding-001)
    Dataset prepared: 1 samples


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

    Evaluation completed!
 Saved detailed results to /content/drive/MyDrive/RAG_Internship/ragas_results_test_20260620_052820.csv

 Test successful!
   Results: {'faithfulness': 1.0000, 'answer_relevancy': 0.7853, 'context_precision': 1.0000, 'context_recall': 1.0000}


Test with 2 Questions

In [31]:
try:
    # Check if we have questions loaded
    if 'test_questions' not in locals() or not test_questions:
        print(" Loading evaluation data...")
        test_questions, ground_truths = load_evaluation_data()

    if test_questions:
        print(" Running evaluation with 2 questions...")
        print(f" Testing with first 2 questions:")
        for i, q in enumerate(test_questions[:2]):
            print(f"   {i+1}. {q}")
        print("-" * 60)

        # Run evaluation with limit=2
        run_full_evaluation_flow(limit=2)

        print("\n 2-question test complete!")

    else:
        print(" No test data available.")
        print("   Please ensure arrays.txt exists with test_questions and ground_truths.")

except Exception as e:
    print(f" Evaluation Error: {e}")
    import traceback
    traceback.print_exc()

 Running evaluation with 2 questions...
 Testing with first 2 questions:
   1. What is machine learning?
   2. What is deep learning?
------------------------------------------------------------

 Evaluating 2 questions across 2 configurations...
   Evaluator: Groq (llama-3.3-70b-versatile)
------------------------------------------------------------

 Evaluating Configuration: Groq-Llama-70B
 Processing 1/2: What is machine learning?...
 Processing 2/2: What is deep learning?...
 Collected predictions for 2 questions.
 Starting RAGAS evaluation for Groq-Llama-70B...
    Evaluator LLM: Groq (llama-3.3-70b-versatile)
    Embeddings: Gemini (models/gemini-embedding-001)
    Dataset prepared: 2 samples


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

    Evaluation completed!
 Saved detailed results to /content/drive/MyDrive/RAG_Internship/ragas_results_groq-llama-70b_20260620_170338.csv

 Evaluating Configuration: Gemini-2.5-Flash
 Processing 1/2: What is machine learning?...
 Processing 2/2: What is deep learning?...
 Collected predictions for 2 questions.
 Starting RAGAS evaluation for Gemini-2.5-Flash...
    Evaluator LLM: Groq (llama-3.3-70b-versatile)
    Embeddings: Gemini (models/gemini-embedding-001)
    Dataset prepared: 2 samples


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[4]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kt1j8e46ewx9agv3pemjjxhb` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 98819, Requested 2610. Please try again in 20m34.656s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
ERROR:ragas.executor:Exception raised in Job[7]: TimeoutError()


    Evaluation completed!
 Saved detailed results to /content/drive/MyDrive/RAG_Internship/ragas_results_gemini-2.5-flash_20260620_170642.csv

 RAGAS CONFIGURATION SUMMARY

🔹 Groq-Llama-70B:
   {'faithfulness': 1.0000, 'answer_relevancy': 0.7897, 'context_precision': 1.0000, 'context_recall': 1.0000}

🔹 Gemini-2.5-Flash:
   {'faithfulness': 1.0000, 'answer_relevancy': 0.7897, 'context_precision': 1.0000, 'context_recall': 1.0000}


 2-question test complete!


In [32]:
# Cell: Quick display of results
import pandas as pd
import glob

try:
    result_files = glob.glob(f"{PROJECT_DIR}/ragas_results_*.csv")

    print("📊 RESULTS SUMMARY OF TWO QUESTIONS")
    print("-" * 60)

    for file in sorted(result_files):
        name = file.split('/')[-1].replace('ragas_results_', '').replace('.csv', '')
        df = pd.read_csv(file)

        avg_scores = {
            'Faithfulness': df['faithfulness'].mean(),
            'Relevancy': df['answer_relevancy'].mean(),
            'Precision': df['context_precision'].mean(),
            'Recall': df['context_recall'].mean()
        }
        avg_scores['Overall'] = sum(avg_scores.values()) / 4

        print(f"\n🔹 {name}")
        for metric, value in avg_scores.items():
            print(f"   {metric}: {value:.4f}")

    print("\n" + "=" * 60)

except Exception as e:
    print(f" Error: {e}")

📊 RESULTS SUMMARY OF TWO QUESTIONS
------------------------------------------------------------

🔹 gemini-2.5-flash_20260619_155947
   Faithfulness: 1.0000
   Relevancy: 0.7897
   Precision: 1.0000
   Recall: 1.0000
   Overall: 0.9474

🔹 groq-llama-70b_20260620_170338
   Faithfulness: 1.0000
   Relevancy: 0.7897
   Precision: 1.0000
   Recall: 1.0000
   Overall: 0.9474

🔹 test_20260619_153823
   Faithfulness: 1.0000
   Relevancy: nan
   Precision: 1.0000
   Recall: 1.0000
   Overall: nan

